# Ejecuciones y Resultados

<div style="text-align: justify;">

Este _notebook_ ejecuta las variantes del _NSGA-II_ con los hiperparámetros obtenidos en la parametrización (`Results/Parametrizacion`) sobre los datasets de evaluación (`B1`, `B2`, `B3`), usando **31 semillas** derivadas de la fecha del día y compartidas por todas las variantes. Cada ejecución se guarda de forma incremental (CSV de historial + frente F0 en `.npz`), por lo que una sesión interrumpida se reanuda sin repetir lo ya ejecutado.

</div>

In [1]:
# 1. Clonar el repo
#!git clone https://github.com/BenjaminGonzalezH/Thesis.git /content/Thesis

# 2. Montar Drive
#from google.colab import drive
#drive.mount('/content/drive')

# 3. Copiar las matrices desde Drive al directorio que usa el notebook
import shutil
from pathlib import Path

DRIVE_MATRICES = Path("/content/drive/MyDrive/")                   # <-- ajusta a tu carpeta en Drive
DEST_MATRICES  = Path("/content/Thesis/Datasets/Matrices")        # minúscula: así lo espera el notebook en Colab

DEST_MATRICES.mkdir(parents=True, exist_ok=True)

archivos = list(DRIVE_MATRICES.glob("*.npz"))
for f in archivos:
    shutil.copy2(f, DEST_MATRICES / f.name)

print(f"{len(archivos)} archivos copiados a {DEST_MATRICES}")
print(sorted(p.name for p in DEST_MATRICES.iterdir()))

0 archivos copiados a \content\Thesis\Datasets\Matrices
['A1_S_DB.npz', 'A1_S_DE.npz', 'A2_S_DB.npz', 'A2_S_DB_M.npz', 'A2_S_DE.npz', 'B1_S_DB.npz', 'B1_S_DB_M.npz', 'B1_S_DE.npz', 'B2_S_DB.npz', 'B2_S_DB_M.npz', 'B2_S_DE.npz', 'B3_S_DB.npz', 'clustered_heatmap_BI_A1.png', 'clustered_heatmap_BI_A1_M.png', 'clustered_heatmap_BI_A2.png', 'clustered_heatmap_BI_A2_M.png', 'clustered_heatmap_BI_B1.png', 'clustered_heatmap_BI_B1_M.png', 'clustered_heatmap_BI_B2.png', 'clustered_heatmap_BI_B2_M.png', 'clustered_heatmap_BI_B3.png', 'clustered_heatmap_BI_B3_M.png', 'clustered_heatmap_GE_A1.png', 'clustered_heatmap_GE_A2.png', 'clustered_heatmap_GE_B1.png', 'clustered_heatmap_GE_B2.png', 'clustered_heatmap_GE_B3.png']


## 1. Ambiente y semillas

In [2]:
#######################################################
# MODULO DE PREPARACIÓN DEL AMBIENTE
# Detección de entorno (Colab / Local), rutas del repositorio, respaldo en
# Drive y semillas de las ejecuciones (31, derivadas de la fecha del día).
#######################################################

############################
# Importaciones
############################
import json                                                 # Guardado de las semillas y de las configuraciones.
import sys                                                  # Ajuste de sys.path para importar los módulos de Source Code.
import os                                                   # Llamadas al SO.
import shutil                                               # Copia de archivos entre el disco local y Drive.
from datetime import date                                   # Semilla basada en la fecha del día.
import numpy  as np                                         # Aplicación de operaciones matematicas eficientes.
import pandas as pd                                         # Historial y resumen de resultados.
from pathlib import Path                                    # Aplicar un formato generalizado de direcciones en el script.

############################
# Detección de entorno (Colab / Local)
############################
try:
    import google.colab                                     # Módulo disponible solo dentro de runtimes de Colab.
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    THESIS_DIR = Path("/content/Thesis")
else:
    _cwd = Path(os.getcwd()).resolve()
    THESIS_DIR = next(
        (p for p in (_cwd, *_cwd.parents) if (p / "Datasets").is_dir() and (p / "Source Code").is_dir()),
        None,
    )
    if THESIS_DIR is None:
        raise FileNotFoundError(f"No se encontró la raíz del repositorio (Datasets/ y Source Code/) desde {_cwd}")

MATRICES_DIR = THESIS_DIR / "Datasets" / "Matrices"
print(f"[entorno] {'Colab' if IN_COLAB else 'Local'} → {THESIS_DIR}")

SOURCE_CODE_DIR = THESIS_DIR / "Source Code"
if str(SOURCE_CODE_DIR) not in sys.path:
    sys.path.append(str(SOURCE_CODE_DIR))

############################
# Directorios de resultados y respaldo en Drive
############################
PARAM_DIR   = THESIS_DIR / "Results" / "Parametrizacion"   # Hiperparámetros obtenidos en la parametrización.
RESULTS_DIR = THESIS_DIR / "Results" / "Ejecuciones"       # Resultados de este notebook.
FRENTES_DIR = RESULTS_DIR / "Frentes"                      # Un .npz por ejecución (F0, medoides e historial de HV).
FRENTES_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis_Results/Ejecuciones")    # <-- ajusta a tu carpeta en Drive
DRIVE_PARAM_DIR   = Path("/content/drive/MyDrive/Thesis_Results/Parametrizacion")

def _copiar_arbol(origen: Path, destino: Path, restaurar: bool) -> None:
    """Copia recursiva de `origen` a `destino`. Con `restaurar`, no pisa archivos existentes; si no, solo copia lo nuevo o modificado."""
    if not origen.exists():
        return
    for archivo in origen.rglob("*"):
        if not archivo.is_file():
            continue
        objetivo = destino / archivo.relative_to(origen)
        if objetivo.exists() and (restaurar or objetivo.stat().st_mtime >= archivo.stat().st_mtime):
            continue
        objetivo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archivo, objetivo)

def sincronizar_drive(restaurar: bool = False) -> None:
    """Local -> Drive (o Drive -> local si `restaurar`). No hace nada fuera de Colab."""
    if not IN_COLAB:
        return
    try:
        if restaurar:
            _copiar_arbol(DRIVE_PARAM_DIR,   PARAM_DIR,   restaurar=True)
            _copiar_arbol(DRIVE_RESULTS_DIR, RESULTS_DIR, restaurar=True)
        else:
            _copiar_arbol(RESULTS_DIR, DRIVE_RESULTS_DIR, restaurar=False)
    except Exception as e:                                  # Un fallo de Drive no debe interrumpir las ejecuciones.
        print(f"[drive] no se pudo {'restaurar' if restaurar else 'respaldar'}: {e}")

sincronizar_drive(restaurar=True)

############################
# Semillas (31, derivadas de la fecha del día AAAAMMDD)
############################
N_SEEDS = 31
SEEDS_FILE = RESULTS_DIR / "seeds_ejecuciones.json"

SEED = int(date.today().strftime("%Y%m%d"))                 # Semilla del día: origen de las 31 semillas.
SEEDS = [int(s) for s in np.random.default_rng(SEED).integers(0, 2**31 - 1, size=N_SEEDS)]

# Persistencia: si ya existen semillas guardadas (sesión previa), se reutilizan para que
# la serie de ejecuciones sea siempre la misma aunque el notebook se reanude otro día.
if SEEDS_FILE.exists():
    with open(SEEDS_FILE, "r", encoding="utf-8") as f:
        _guardado = json.load(f)
    SEED, SEEDS = _guardado["seed"], _guardado["seeds"]
    print(f"[semillas] cargadas desde {SEEDS_FILE.name} (generadas el {_guardado['fecha']})")
else:
    with open(SEEDS_FILE, "w", encoding="utf-8") as f:
        json.dump({"fecha": date.today().isoformat(), "seed": SEED, "seeds": SEEDS}, f, indent=2)
    print(f"[semillas] archivo creado: {SEEDS_FILE.name}")
print(f"[semillas] SEED = {SEED} | {len(SEEDS)} semillas: {SEEDS[:5]} ...")
print(f"[resultados] {RESULTS_DIR}")

[entorno] Local → C:\Users\benja\Desktop\workspace\Thesis
[semillas] cargadas desde seeds_ejecuciones.json (generadas el 2026-11-01)
[semillas] SEED = 20261101 | 31 semillas: [1183181808, 1270388439, 1658144228, 768834435, 978234145] ...
[resultados] C:\Users\benja\Desktop\workspace\Thesis\Results\Ejecuciones


## 2. Código fuente e hiperparámetros

In [3]:
#######################################################
# MODULO DE IMPORTACIÓN DEL CÓDIGO FUENTE E HIPERPARÁMETROS
#######################################################

import solution_encoding                                    # OBJ_FUNCTION_CALLS es un contador global: se reinicia antes de cada ejecución.
from solution_encoding import load_distance_matrix
from LS import build_neighborhood_matrix                    # M_V de MOLS: GE, BI o BOTH.
from nsga2 import run_nsga2
from performance import jaccard_f0_history, jaccard_population_history   # Jaccard de F0 y de la población, por generación.
from nsga2_variants import run_nsga2_MOPR, run_nsga2_PLS, run_nsga2_MOLS
from nsga2_variants_bio import run_nsga2_PNS, run_nsga2_DBS

VARIANTES = {
    "NSGA2":       run_nsga2,
    "NSGA2_PLS":   run_nsga2_PLS,
    "NSGA2_MOLS":  run_nsga2_MOLS,
    "NSGA2_DBS":   run_nsga2_DBS,
    "NSGA2_PNS":   run_nsga2_PNS,
    "NSGA2_MOPR":  run_nsga2_MOPR,
}

############################
# Mejores hiperparámetros de la parametrización
############################
def cargar_mejores_parametros() -> dict:
    """
    Lee `mejores_parametros.json`. Si no existe (la parametrización no llegó a escribirlo),
    se reconstruye desde los estudios de Optuna (`<variante>.db`).
    """
    archivo = PARAM_DIR / "mejores_parametros.json"
    if archivo.exists():
        with open(archivo, "r", encoding="utf-8") as f:
            return json.load(f)

    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    mejores = {}
    for variante in VARIANTES:
        db = PARAM_DIR / f"{variante}.db"
        if not db.exists():
            raise FileNotFoundError(f"Sin parámetros para {variante}: falta {archivo.name} y {db.name} en {PARAM_DIR}")
        estudio = optuna.load_study(study_name=variante, storage=f"sqlite:///{db}")
        mejores[variante] = {"hv_f1": estudio.best_value, "parametros": estudio.best_params}
    print(f"[parámetros] reconstruidos desde los estudios .db ({PARAM_DIR})")
    return mejores

MEJORES = cargar_mejores_parametros()
for v in VARIANTES:
    print(f"  {v:<11} HV param = {MEJORES[v]['hv_f1']:.4f} | {MEJORES[v]['parametros']}")

[backend] CuPy detectado → operaciones vectorizadas en GPU.
[backend] CuPy detectado → operaciones vectorizadas en GPU.
  NSGA2       HV param = 14284.6514 | {'pop_size': 164, 'crossover_prob': 0.8400000000000001, 'mutation_prob': 0.05}
  NSGA2_PLS   HV param = 14293.8059 | {'pop_size': 53, 'crossover_prob': 0.8700000000000001, 'mutation_prob': 0.01, 'pls_neighborhood': 0.13, 'pls_local_budget': 9200}
  NSGA2_MOLS  HV param = 14282.7604 | {'pop_size': 232, 'crossover_prob': 0.88, 'mutation_prob': 0.04, 'M_v': 'BOTH', 'mode': 'large', 'mols_local_budget': 8700, 'neighborhood_pct': 31}
  NSGA2_DBS   HV param = 14285.6794 | {'pop_size': 297, 'crossover_prob': 0.9, 'mutation_prob': 0.05, 'mode': 'expresion', 'p_pct': 8}
  NSGA2_PNS   HV param = 14293.1220 | {'pop_size': 86, 'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.02, 'nF': 3}
  NSGA2_MOPR  HV param = 14287.9753 | {'pop_size': 250, 'crossover_prob': 0.89, 'mutation_prob': 0.05}


## 3. Cuadro de ejecución

Define qué variantes, datasets y semillas se ejecutan. Los resultados quedan en `Results/Ejecuciones`:

| Archivo | Contenido |
|---|---|
| `historial_ejecuciones.csv` | Registro acumulativo (una fila por ejecución: variante, dataset, semilla, hipervolumen inicial/final, generaciones, llamadas, tiempo, fecha y parámetros). |
| `Frentes/<variante>_<dataset>_<semilla>.npz` | Frente F0 (objetivos y medoides) e historial de hipervolumen de la ejecución. |
| `seeds_ejecuciones.json` | Las 31 semillas usadas. |

Las ejecuciones ya registradas (misma variante, dataset y semilla) se omiten, salvo que `REEJECUTAR = True`; en ese caso se agrega una fila nueva al historial y el resumen usa la más reciente.

In [ ]:
#######################################################
# MODULO DE EJECUCIÓN
# Ejecuta variante × dataset × semilla con los mejores hiperparámetros,
# guardando cada ejecución apenas termina (CSV de historial + .npz del frente).
#######################################################

import io                                                   # Silenciar el log por generación de los algoritmos.
import time                                                 # Tiempo por ejecución.
import contextlib                                           # redirect_stdout.
from datetime import datetime

############################
# Configuración del usuario
############################
VARIANTES_A_EJECUTAR = list(VARIANTES)                      # Subconjunto, p. ej. ["NSGA2", "NSGA2_PNS"].
DATASETS_A_EJECUTAR  = ["B1", "B2", "B3"]                   # Datasets de evaluación (prefijo B).
SEEDS_A_EJECUTAR     = SEEDS                                # Subconjunto, p. ej. SEEDS[:5] para una prueba.
REEJECUTAR           = False                                # True: repite ejecuciones ya registradas (se agrega al historial).

############################
# Constantes del experimento
############################
K                      = 4                                  # Medoides por solución.
MAX_OBJ_FUNCTION_CALLS = 250_000                            # Presupuesto POR ejecución.
HISTORIAL_FILE         = RESULTS_DIR / "historial_ejecuciones.csv"
assert set(VARIANTES_A_EJECUTAR) <= set(VARIANTES), "Variante desconocida en VARIANTES_A_EJECUTAR."

############################
# Datasets (carga bajo demanda: B puede ser muy grande)
############################
_CACHE = {}

def cargar_dataset(nombre: str) -> dict:
    """Carga GE/BI. Los M_V y sus radios se calculan solo cuando una configuración de MOLS los requiere."""
    if nombre not in _CACHE:
        _CACHE.clear()                                      # Un solo dataset en memoria a la vez.
        ge, _ = load_distance_matrix(MATRICES_DIR / f"{nombre}_S_DE.npz")
        bi, _ = load_distance_matrix(MATRICES_DIR / f"{nombre}_S_DB_M.npz")
        _CACHE[nombre] = {"ge": ge, "bi": bi, "n": ge.shape[0], "m_v": {}, "radios": {}}
    return _CACHE[nombre]

def radio_mols(ds: dict, fuente: str, percentil: int) -> tuple:
    """M_V y radio `neighborhood` (percentil de M_V sin diagonal), con memoización por dataset."""
    if fuente not in ds["m_v"]:
        ds["m_v"][fuente] = build_neighborhood_matrix(ds["ge"], ds["bi"], fuente)
    clave = (fuente, percentil)
    if clave not in ds["radios"]:
        m_v = ds["m_v"][fuente]
        ds["radios"][clave] = float(np.percentile(m_v[~np.eye(ds["n"], dtype=bool)], percentil))
    return ds["m_v"][fuente], ds["radios"][clave]

############################
# Argumentos de cada variante (misma traducción que en Parametrización)
############################
def kwargs_variante(variante: str, cfg: dict, ds: dict, seed: int) -> dict:
    kw = dict(
        ge_matrix=ds["ge"], bi_matrix=ds["bi"], n=ds["n"], k=K,
        pop_size=cfg["pop_size"], max_obj_function_calls=MAX_OBJ_FUNCTION_CALLS,
        crossover_prob=cfg["crossover_prob"], mutation_prob=cfg["mutation_prob"],
        seed=seed,
    )
    if variante == "NSGA2_PLS":
        kw["pls_neighborhood"] = cfg["pls_neighborhood"]
        kw["pls_local_budget"] = cfg["pls_local_budget"]
    elif variante == "NSGA2_MOLS":
        m_v, radio = radio_mols(ds, cfg["M_v"], cfg["neighborhood_pct"])
        kw["mode"]              = cfg["mode"]
        kw["m_v_matrix"]        = m_v
        kw["neighborhood"]      = radio
        kw["mols_local_budget"] = cfg["mols_local_budget"]
    elif variante == "NSGA2_PNS":
        kw["nF"]               = cfg["nF"]
    elif variante == "NSGA2_DBS":
        kw["mode"]             = cfg["mode"]
        kw["p"]                = cfg["p_pct"] / 100
    return kw

############################
# Historial
############################
def leer_historial() -> pd.DataFrame:
    return pd.read_csv(HISTORIAL_FILE) if HISTORIAL_FILE.exists() else pd.DataFrame()

def registrar(fila: dict) -> None:
    pd.DataFrame([fila]).to_csv(HISTORIAL_FILE, mode="a", header=not HISTORIAL_FILE.exists(), index=False)

def ejecutadas() -> set:
    h = leer_historial()
    return set() if h.empty else set(zip(h["variante"], h["dataset"], h["semilla"]))

############################
# Ejecución de una corrida
############################
def ejecutar(variante: str, nombre_ds: str, semilla: int) -> dict:
    cfg = MEJORES[variante]["parametros"]
    ds  = cargar_dataset(nombre_ds)

    solution_encoding.OBJ_FUNCTION_CALLS = 0                # Reinicio obligatorio del presupuesto.
    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):         # Se silencia el print por generación.
        res = VARIANTES[variante](**kwargs_variante(variante, cfg, ds, semilla))
    segundos = time.time() - t0

    hv = res["history_hv_f1"]
    jac = jaccard_f0_history(res["history_objectives"], res["history_labels"])   # Diversidad de F0 por generación (NaN si |F0| < 2).
    jac_pop = jaccard_population_history(res["history_labels"])                  # Diversidad de la población completa por generación.
    f0 = np.asarray(res["fronts"][0])
    np.savez_compressed(
        FRENTES_DIR / f"{variante}_{nombre_ds}_{semilla}.npz",
        objectives=np.asarray(res["objectives"])[f0],
        medoids=np.asarray(res["population"])[f0],
        history_hv_f1=np.asarray(hv, dtype=float),
        history_jaccard_f0=np.asarray(jac, dtype=float),
        history_jaccard_pop=np.asarray(jac_pop, dtype=float),
    )
    fila = {
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "variante": variante, "dataset": nombre_ds, "semilla": semilla,
        "hv_inicial": float(hv[0]), "hv_final": float(hv[-1]),
        "tam_f0": int(len(f0)), "generaciones": int(res["generations_run"]),
        "llamadas": int(solution_encoding.OBJ_FUNCTION_CALLS), "segundos": round(segundos, 1),
        "parametros": json.dumps(cfg, ensure_ascii=False),
    }
    registrar(fila)
    sincronizar_drive()                                     # Respaldo en Drive tras cada ejecución (solo Colab).
    return fila

############################
# Cuadro de ejecución (dataset como bucle externo: evita recargar matrices grandes)
############################
hechas = set() if REEJECUTAR else ejecutadas()
total  = len(VARIANTES_A_EJECUTAR) * len(DATASETS_A_EJECUTAR) * len(SEEDS_A_EJECUTAR)
print(f"[plan] {total} ejecuciones en total | {len(hechas)} ya registradas")

contador = 0
for nombre_ds in DATASETS_A_EJECUTAR:
    for variante in VARIANTES_A_EJECUTAR:
        for semilla in SEEDS_A_EJECUTAR:
            contador += 1
            if (variante, nombre_ds, semilla) in hechas:
                continue
            fila = ejecutar(variante, nombre_ds, semilla)
            print(f"[{contador:>4}/{total}] {variante:<11} {nombre_ds} semilla={semilla:<11} "
                  f"HV F0 = {fila['hv_final']:.4f} | |F0| = {fila['tam_f0']:>3} | {fila['segundos'] / 60:.1f} min")
print("\n[fin] ejecuciones completadas")

[plan] 558 ejecuciones en total | 0 ya registradas
[load] 'B1_S_DE.npz' → matriz 3187×3187. Primer gen: 'CDH2', último: 'ENSG00000285973'.
[load] 'B1_S_DB_M.npz' → matriz 3187×3187. Primer gen: 'CDH2', último: 'ENSG00000285973'.


C:\Users\benja\Desktop\workspace\Thesis\Source Code\pareto_sorting.py:97: RuntimeWarning: invalid value encountered in scalar divide
  contribution = (next_val - prev_val) / denom


[   1/558] NSGA2       B1 semilla=1183181808  HV F0 = 14263.2734 | |F0| =  14 | 1.5 min
[   2/558] NSGA2       B1 semilla=1270388439  HV F0 = 14265.6654 | |F0| =  21 | 1.6 min
[   3/558] NSGA2       B1 semilla=1658144228  HV F0 = 14265.7834 | |F0| =   9 | 1.6 min
[   4/558] NSGA2       B1 semilla=768834435   HV F0 = 14261.3344 | |F0| =  18 | 1.6 min
[   5/558] NSGA2       B1 semilla=978234145   HV F0 = 14261.3284 | |F0| =  11 | 1.6 min
[   6/558] NSGA2       B1 semilla=1039227980  HV F0 = 14263.5790 | |F0| =  15 | 1.6 min
[   7/558] NSGA2       B1 semilla=245761548   HV F0 = 14262.2188 | |F0| =  22 | 1.5 min
[   8/558] NSGA2       B1 semilla=2118637595  HV F0 = 14262.5240 | |F0| =  14 | 1.6 min
[   9/558] NSGA2       B1 semilla=358181644   HV F0 = 14262.6887 | |F0| =  13 | 1.6 min
[  10/558] NSGA2       B1 semilla=291931378   HV F0 = 14268.4155 | |F0| =  13 | 1.6 min
[  11/558] NSGA2       B1 semilla=2116980539  HV F0 = 14264.3611 | |F0| =  16 | 1.7 min
[  12/558] NSGA2       B1 semill

## 4. Resultados

Resumen del hipervolumen final de F0 por variante y dataset (usa la ejecución más reciente de cada combinación).

In [5]:
#######################################################
# MODULO DE RESUMEN DE RESULTADOS
#######################################################

historial = leer_historial()
if historial.empty:
    print("Aún no hay ejecuciones registradas.")
else:
    ultimas = historial.sort_values("fecha").drop_duplicates(["variante", "dataset", "semilla"], keep="last")

    resumen = (ultimas.groupby(["dataset", "variante"])
               .agg(n=("hv_final", "size"), hv_media=("hv_final", "mean"), hv_desv=("hv_final", "std"),
                    hv_min=("hv_final", "min"), hv_max=("hv_final", "max"),
                    tam_f0_medio=("tam_f0", "mean"), minutos_medio=("segundos", lambda s: s.mean() / 60))
               .round(4))
    resumen.to_csv(RESULTS_DIR / "resumen_resultados.csv")
    sincronizar_drive()
    display(resumen)

    # Matriz semilla × variante (HV final) por dataset: base para las pruebas estadísticas.
    for nombre_ds, g in ultimas.groupby("dataset"):
        print(f"\n{nombre_ds}: HV final por semilla")
        display(g.pivot(index="semilla", columns="variante", values="hv_final").round(4))

n    hv_media  hv_desv      hv_min      hv_max  \
dataset variante                                                      
B1      NSGA2       31  14263.8089   2.7942  14258.8871  14269.6039   
        NSGA2_DBS   31  14264.9236   3.4829  14257.4137  14270.7402   
        NSGA2_MOLS  31  14263.8327   5.9879  14253.1017  14275.3340   
        NSGA2_MOPR  31  14264.7864   3.4550  14258.9365  14270.2743   
        NSGA2_PLS   31  14272.5384   1.3343  14269.4247  14275.4968   
        NSGA2_PNS   31  14270.3451   2.4315  14261.9182  14274.2500   
B2      NSGA2       31  14285.5169   2.0654  14281.9625  14290.1847   
        NSGA2_DBS   31  14286.9445   2.1380  14282.7256  14291.5207   
        NSGA2_MOLS  31  14284.2074   2.9663  14277.5609  14289.7507   
        NSGA2_MOPR  31  14286.6812   2.4369  14282.4677  14292.9701   
        NSGA2_PLS   31  14291.9507   1.9353  14288.1594  14294.6252   
        NSGA2_PNS   31  14287.0428   3.3799  14277.8497  14292.0969   
B3      NSGA2       31  14264.7783   1.6191  14261.7605  14269.4828   
        NSGA2_DBS   31  14267.5269   1.6560  14263.9520  14270.2753   
        NSGA2_MOLS  31  14267.8039   2.3767  14263.4776  14272.9796   
        NSGA2_MOPR  31  14264.2549   2.6794  14259.6883  14270.5593   
        NSGA2_PLS   31  14269.9863   1.7560  14267.1362  14273.0903   
        NSGA2_PNS   31  14269.3917   2.2206  14265.8441  14274.4163   

                    tam_f0_medio  minutos_medio  
dataset variante                                 
B1      NSGA2            14.2258         0.7679  
        NSGA2_DBS        13.0968         0.4301  
        NSGA2_MOLS       22.8710         0.1111  
        NSGA2_MOPR       14.7742         0.8687  
        NSGA2_PLS        18.7419         0.1585  
        NSGA2_PNS        14.7419         0.2395  
B2      NSGA2            15.4516         0.7558  
        NSGA2_DBS        12.8710         0.4428  
        NSGA2_MOLS       22.9677         0.1280  
        NSGA2_MOPR       17.5806         0.8116  
        NSGA2_PLS        23.1935         0.1525  
        NSGA2_PNS        17.2581         0.2381  
B3      NSGA2            27.7097         1.9592  
        NSGA2_DBS        18.2903         0.2968  
        NSGA2_MOLS       33.0323         0.2125  
        NSGA2_MOPR       26.3548         2.0559  
        NSGA2_PLS        28.8710         0.4919  
        NSGA2_PNS        34.3871         0.3728


B1: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,
2241395,14268.9789,14270.7402,14253.1017,14270.2743,14273.7621,14270.4307
149968717,14260.6927,14262.9622,14272.6112,14259.3344,14275.0891,14270.7432
245761548,14262.2188,14264.3308,14258.6842,14267.6214,14271.7167,14268.0168
291931378,14264.4491,14257.8940,14256.4052,14260.6621,14271.7060,14272.7260
346868666,14267.1590,14263.0056,14263.2666,14263.1820,14271.4567,14268.1683
358181644,14262.6887,14257.4137,14270.9066,14267.6551,14273.0100,14261.9182
398383450,14260.4368,14266.1059,14266.0361,14258.9365,14272.7234,14270.3834
463504515,14260.8092,14263.5045,14269.9638,14260.3548,14273.0230,14268.3054
536623952,14264.7034,14264.0959,14260.0825,14268.0665,14274.1241,14270.0150



B2: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,
2241395,14284.7280,14288.1775,14278.4604,14283.8956,14292.5378,14283.6684
149968717,14285.6907,14288.9798,14285.7895,14286.1647,14292.3072,14290.4042
245761548,14283.6471,14286.0512,14284.7298,14285.8379,14294.6252,14288.9490
291931378,14283.5726,14284.0525,14286.0277,14282.4677,14290.2764,14291.0399
346868666,14286.0504,14288.0577,14287.9019,14283.7574,14289.3925,14289.2975
358181644,14284.0046,14283.1776,14286.0075,14285.9707,14288.1594,14289.3071
398383450,14283.0595,14282.7256,14280.2542,14284.4652,14293.5221,14286.1101
463504515,14286.7057,14285.5218,14281.6646,14286.1546,14289.4043,14289.1072
536623952,14283.2671,14284.8751,14287.2045,14288.2598,14293.2475,14280.6826



B3: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,
2241395,14264.2096,14264.6488,14269.3868,14265.0882,14267.1362,14272.5896
149968717,14264.1033,14267.7181,14266.3748,14263.8880,14271.6613,14268.6583
245761548,14265.6193,14266.4795,14268.5103,14267.0676,14270.2519,14269.1986
291931378,14264.7194,14269.6410,14266.7067,14263.3160,14267.3565,14272.1795
346868666,14265.2060,14267.6771,14272.9796,14259.6883,14270.0395,14268.1383
358181644,14267.1207,14266.4320,14271.0875,14264.2250,14269.4455,14267.0270
398383450,14264.3077,14269.3100,14268.8260,14260.3508,14270.6960,14270.3457
463504515,14264.3589,14265.9966,14268.5788,14262.7081,14268.0945,14270.1833
536623952,14267.2114,14263.9520,14265.6035,14270.1050,14270.6676,14268.8042
